# pipeline_v4's sheet without the video, on batch2_A: exploratory

**Exploratory, not pre-registered.** `prereg_batch2_A.md` registered two conditions on batch2_A's 798 questions,
`A_video` and `pipeline_v4` (video + sheet); their results are `results/final2/`. This notebook adds a third, run
after seeing them, so nothing it shows tests the registered hypothesis:

- **`pipeline_v4_no_video`**: pipeline_v4's sheet, word for word as final2 gave it, and the question, without the video
  (`--no-video`): the final2 prompt less its video and its first line ("These are frames of a video.").

The sheet is written by `v4_eval.py` with b381205's renderer (`scene_sheet_b381205.py`), as final2 ran it; for the 798
questions the prompt with the video equals b381205's (checked 2026-11-24). Same model and settings as final2:
Qwen3.5-2B, thinking off, the sheet's frames `uniform128` (16), up to 2,048 new tokens, final answer scored.

The summary sets it beside final2's two runs, task by task, and gives the input tokens of each.

About 15–20 minutes on an L4: installing about 5, the 38 videos about 5 (only to read each video's fps, which the
sheet's `first_seen` timestamps are written in, as final2 did), the run about 5.

**After a crash**, run it again from the top: the run continues from its file on Drive if this notebook started it.

**Files on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-v4-sheet-only.zip`, the code, made in the repository with

```
git archive --format=zip -o mvp-3d-vlm-v4-sheet-only.zip COMMIT src experiments/2026-11-18-scene-sheet-formats requirements.txt pyproject.toml
```

and `scene_sheets_pipeline_v4_batch2_A.zip` (final2's, checked by its registered SHA-256). Predictions go to
`MyDrive/mvp/v4_sheet_only_results/`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-v4-sheet-only.zip"
V4_ZIP = f"{DRIVE_DIR}/scene_sheets_pipeline_v4_batch2_A.zip"
V4_ZIP_SHA256 = "1003f1d0691da771c7cd8232046551d04ee88cf0229654aff52e0913f799cf9f"  # prereg_batch2_A.md
RESULTS_DIR = f"{DRIVE_DIR}/v4_sheet_only_results"
WORK = "/content/mvp-3d-vlm"
EXP = f"{WORK}/experiments/2026-11-18-scene-sheet-formats"
IDS_PATH = f"{EXP}/batch2_A_ids.txt"
FINAL2 = f"{EXP}/results/final2"      # A_video and pipeline_v4, the registered runs
V4_ROOT = f"{WORK}/data/scene_sheets_pipeline_v4"
VSI_REPO, VSI_REVISION = "nyu-visionx/VSI-Bench", "3a6a1cf9ac54c403509b15260897cc6ddc41b408"
VSI_SHA256 = "aa3ff8cc2010e555f4f865ff96483812ec0b275005b68562f53bb114ad4db9db"
MODEL = "Qwen/Qwen3.5-2B"
COMMON = ["--model", MODEL, "--nframes", "16", "--sampling", "uniform128", "--max-new-tokens", "2048",
          "--final-answer", "--ids", IDS_PATH]
NAME = "pipeline_v4_no_video"

import hashlib, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
shutil.rmtree(WORK, ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(WORK)
    CODE_COMMIT = z.comment.decode() or "unknown"  # git archive writes the commit as the zip's comment
needs = {"src/mvp/eval/vsibench.py": "--no-video", "experiments/2026-11-18-scene-sheet-formats/v4_eval.py": "b381205"}
old = [f for f, text in needs.items() if not os.path.exists(f"{WORK}/{f}") or text not in open(f"{WORK}/{f}").read()]
assert not old, f"{CODE_ZIP} lacks {old}: make it again as the first cell says"
assert hashlib.sha256(Path(V4_ZIP).read_bytes()).hexdigest() == V4_ZIP_SHA256, f"{V4_ZIP} is not final2's sheets"
os.makedirs(RESULTS_DIR, exist_ok=True)

reqs = [l.split("#")[0].strip() for l in open(f"{WORK}/requirements.txt").read().splitlines()]
reqs = [r for r in reqs if r and not r.startswith("--extra-index-url") and r.split("==")[0] not in ("torch", "torchvision")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)  # same module, newer Pythons
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK], check=True)
sys.path.insert(0, f"{WORK}/src")
ENV = {**os.environ, "PYTHONPATH": f"{WORK}/src"}

import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > GPU (L4 or A100)"
GPU = torch.cuda.get_device_name(0)
print(f"code {CODE_COMMIT}; torch {torch.__version__}; GPU {GPU}")

## Data

As final2: the sheets, `test.jsonl` (pinned and checked), the 38 videos, each video's 128 candidate frames, and its fps
as the eval's video reader reports it (the sheet's `first_seen` timestamps are written with it). Check 1: the 16 frames
are the ones the sheets were made on and the ones final2's runs saw.

In [ ]:
import csv, inspect, json
import numpy as np
from huggingface_hub import HfFileSystem, hf_hub_download

RAW, VGGT = Path(WORK) / "data/raw/VSI-Bench", Path(WORK) / "data/vggt"
with zipfile.ZipFile(V4_ZIP) as z:
    prefix = min((n for n in z.namelist() if n.endswith("index.csv")), key=len)[: -len("index.csv")]
    for name in z.namelist():
        if name.startswith(prefix) and not name.endswith("/"):
            dest = Path(V4_ROOT) / name[len(prefix):]
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(z.read(name))
RAW.mkdir(parents=True, exist_ok=True)
shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset", revision=VSI_REVISION), RAW / "test.jsonl")
assert hashlib.sha256((RAW / "test.jsonl").read_bytes()).hexdigest() == VSI_SHA256, "not the expected test.jsonl"

IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
QUESTIONS = {int(q["id"]): q for q in map(json.loads, open(RAW / "test.jsonl"))}
SCENES = sorted({f"{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}" for q in IDS})
fs = HfFileSystem()
for dataset in sorted({s.split("/")[0] for s in SCENES}):
    todo = [s for s in SCENES if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]
    if todo:
        with fs.open(f"datasets/{VSI_REPO}@{VSI_REVISION}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in todo:
                (RAW / s).parent.mkdir(parents=True, exist_ok=True)
                (RAW / f"{s}.mp4").write_bytes(z.read(f"{s}.mp4"))
print(f"{len(IDS)} questions, {len(SCENES)} videos")

# frame_indices.npy as the VGGT export takes them, and meta.json's video_fps as the eval's reader reports it (final2's
# fps cell): the sheet's first_seen is written in that fps
from decord import VideoReader, cpu
from mvp.benchmarks import scene_sheet as ss
from mvp.models.qwen_vl import QwenVL
from mvp.reconstruction.sa_sampling import uniform_subset
from qwen_vl_utils.vision_process import get_video_reader_backend

reader = QwenVL.__new__(QwenVL)
limits = inspect.signature(QwenVL.__init__).parameters
reader.min_pixels, reader.max_pixels = limits["min_pixels"].default, limits["max_pixels"].default
for s in SCENES:
    n = len(VideoReader(str(RAW / f"{s}.mp4"), ctx=cpu(0)))
    (VGGT / s).mkdir(parents=True, exist_ok=True)
    np.save(VGGT / s / "frame_indices.npy", np.linspace(0, n - 1, num=min(128, n), dtype=int).astype(np.int32))
    _, video_meta, _ = reader.load_video(str(RAW / f"{s}.mp4"), nframes=ss.N_FRAMES)
    (VGGT / s / "meta.json").write_text(json.dumps({"video_fps": float(video_meta["fps"]), "num_video_frames": n,
                                                    "video_reader_backend": get_video_reader_backend()}, indent=1))

# Check 1
A = {r["id"]: r for r in map(json.loads, open(f"{FINAL2}/A_video.jsonl"))}
problems = []
for s in SCENES:
    picked = np.load(VGGT / s / "frame_indices.npy")[uniform_subset(128, 16)].tolist()
    report = json.loads((Path(V4_ROOT) / s.replace("/", "__") / "subset-uniform16/scene/adapter_report.json").read_text())
    if picked != report["source_video_frames"]:
        problems.append(f"{s}: frames {picked}, the sheets' {report['source_video_frames']}")
    for q in IDS:
        if f"{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}" == s and A[q]["frame_indices"] != picked:
            problems.append(f"{q}: final2 saw frames {A[q]['frame_indices']}")
assert not problems, "\n".join(problems[:10])
print("OK: every scene's 16 frames are the sheets' and final2's")

## Prompts

Check 2: the eval's prompt for every question (`--print-prompt`, through `v4_eval.py`, no weights) is the chat template
around the sheet and the question only, with no video, no "These are frames of a video." and the v4 relations where
final2 had them.

In [ ]:
sys.path.insert(0, EXP)
import v4_eval
from mvp.benchmarks.vsibench import build_prompt_parts

cmd = [sys.executable, f"{EXP}/v4_eval.py", *COMMON, "--sheet-source", "pipeline", "--sheet-format", "csv",
       "--sheets-root", V4_ROOT, "--no-video", "--print-prompt", str(len(IDS))]
printed = subprocess.run(cmd, cwd=WORK, env=ENV, capture_output=True, text=True)
assert printed.returncode == 0, printed.stdout[-3000:] + printed.stderr[-3000:]
blocks = printed.stdout.split("===== question ")[1:]
v4_eval.install()
problems = []
for block in blocks:
    q = int(block.split(" ", 1)[0])
    text = block.split("\n", 1)[1].strip("\n")
    _, with_video = build_prompt_parts(QUESTIONS[q], "sit", v4_eval.v4_sheet(q, V4_ROOT, VGGT))  # final2's text
    expected = ss.TEXT_CHAT.format(system=ss.SYSTEM_PROMPT, text=with_video.split("\n", 1)[1])
    if text != expected.strip("\n") or "vision" in text:
        problems.append(q)
assert len(blocks) == len(IDS) and not problems, f"{len(blocks)} prompts; not as expected: {problems[:10]}"
print(f"OK: {len(blocks)} prompts, each final2's sheet and question with no video. Question {IDS[0]}:\n")
print(blocks[0].split("\n", 1)[1][:3000])

## Run

In [ ]:
import time
from huggingface_hub import snapshot_download

out, marker = Path(f"{RESULTS_DIR}/{NAME}.jsonl"), Path(f"{RESULTS_DIR}/{NAME}.started")
stamp = f"{CODE_COMMIT} {V4_ZIP_SHA256}\n"
if out.exists() and not (marker.exists() and marker.read_text() == stamp):
    raise RuntimeError(f"{out} was not started by this notebook with this code and these sheets, and the eval would skip "
                       "every question already in it: delete it (or move it out of RESULTS_DIR) and run this cell again")
marker.write_text(stamp)
t0 = time.time()
proc = subprocess.Popen([sys.executable, f"{EXP}/v4_eval.py", *COMMON, "--sheet-source", "pipeline", "--sheet-format",
                         "csv", "--sheets-root", V4_ROOT, "--no-video", "--out", str(out)],
                        cwd=WORK, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, f"{NAME} failed"
print(f"{NAME}: {(time.time() - t0) / 60:.1f} min on {GPU}")
Path(f"{RESULTS_DIR}/model_revision.txt").write_text(
    f"model: {MODEL}\nrevision: {Path(snapshot_download(MODEL, local_files_only=True)).name}\ncode commit: {CODE_COMMIT}\n"
    f"gpu: {GPU}\nexploratory: not in prereg_batch2_A.md\n")

## Summary (exploratory)

Per task ×100 (the eval's `summarize`): final2's `A_video` and `pipeline_v4` and this run, with question-by-question
differences (better / worse, exact sign test, not corrected: exploratory), and input tokens. final2's runs record only
the batch's padded length (`n_input_tokens`); this run also records each question's own (`n_prompt_tokens`).

In [ ]:
import pandas as pd
from mvp.benchmarks.vsibench import summarize
from mvp.reporting.compare_runs import paired, sign_test

def load(path):
    df = pd.read_json(path, lines=True)
    return df[df["id"].isin(IDS)].drop_duplicates("id").set_index("id").sort_index()

runs = {"A_video": load(f"{FINAL2}/A_video.jsonl"), "pipeline_v4": load(f"{FINAL2}/pipeline_v4.jsonl"),
        NAME: load(f"{RESULTS_DIR}/{NAME}.jsonl")}
assert all(len(r) == len(IDS) for r in runs.values()), {k: len(r) for k, r in runs.items()}
assert (runs[NAME]["n_frames"] == 0).all(), "a question was run with frames"
table = pd.DataFrame({c: summarize(r.reset_index()) for c, r in runs.items()}).round(1)
print(table.to_string())
print()
for ref, other in (("pipeline_v4", NAME), ("A_video", NAME)):
    better, worse, same = paired(runs[ref]["score"], runs[other]["score"])
    print(f"{other} − {ref}: overall {table.loc['overall', other] - table.loc['overall', ref]:+.1f}; "
          f"better / worse / same {better} / {worse} / {same}; sign test p = {sign_test(better, worse):.3g}")
print()
for c, r in runs.items():
    col = "n_prompt_tokens" if "n_prompt_tokens" in r else "n_input_tokens"
    print(f"{c}: input tokens ({col}) mean {r[col].mean():.0f}, median {r[col].median():.0f}")
table.to_csv(f"{RESULTS_DIR}/summary_by_task.csv")
print(f"\nsaved {RESULTS_DIR}/summary_by_task.csv")